# Part 5.1 — Partial-State Training Data Validation

This notebook validates the data infrastructure for **Part 5: the joint policy+value network**. Before writing any training code, we need to confirm:

1. **The FM is well-calibrated enough** to serve as a value label source. Since `terminal_win_prob` in every `SelfPlayRecord` comes from the FM, a miscalibrated FM would corrupt every value label we train on.
2. **The replay buffer has sufficient, balanced coverage** across all 6 pick depths. The value head needs training signal at every stage of the draft, not just the end.
3. **The team-swap augmentation works correctly** — it doubles our value training data and the augmented records are structurally sound.
4. **The win probability distribution makes sense** — centred near 0.5 at every depth (expected, since `terminal_win_prob` is always the FM score of the *final* state, not the partial state).

**Pass criteria summary:**
- FM log-loss < 0.685 → safe to use FM labels for value training
- All 6 pick depths have ≥ 100 records, no NaN win probs
- Augmented records: `my_team ↔ opp_team` swapped, `terminal_win_prob = 1 - original`, `visit_dist = {}`
- Win probs centred near 0.5 and symmetric at every depth

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from bsdraft.fm.evaluate import check_fm_calibration
from bsdraft.mcts.evaluator import FMEvaluator
from bsdraft.fm.model import FMInference
from bsdraft.data.matchup_db import MatchupDB
from bsdraft.selfplay.generate import ReplayBuffer, verify_buffer_coverage, augment_with_team_swap
from bsdraft.data.prep import SEASON_CONFIGS

SEASON   = "s48"
DATA_DIR = Path("..").resolve() / "data" / SEASON

fm        = FMInference.load(DATA_DIR / "fm_model.pkl")
evaluator = FMEvaluator(fm)
db        = MatchupDB.load(DATA_DIR / "matchup_db.pkl")

buf     = ReplayBuffer.load(DATA_DIR / "self_play")
records = buf.all_records()

print(f"Season      : {SEASON}")
print(f"FM          : val_logloss={fm.val_logloss:.4f}  val_auc={fm.val_auc:.4f}  val_brier={fm.val_brier:.4f}")
print(f"Replay buf  : {buf.n_games:,} games  |  {buf.n_records:,} records")


## FM Calibration Check

Every `SelfPlayRecord.terminal_win_prob` is a FM prediction, so the quality of our value training labels is **bounded by FM calibration**. If the FM is poorly calibrated — systematically overconfident or biased — the value head will learn to reproduce those errors.

The threshold is **log-loss < 0.685** (random guessing = 0.693). Below that we have genuine signal; above it the FM is barely better than a coin flip and the value labels are too noisy to train on reliably.

We check the stored training-time metrics — no need to reload the full validation set.

In [ ]:
cal = check_fm_calibration(fm)

if not cal["pass"]:
    print("\n⚠️  FM calibration does not meet the threshold.")
    print("   Value network training will likely produce noisy, unreliable labels.")
    print("   Consider retraining the FM before proceeding to Part 5.2.")
else:
    print("\nFM calibration is sufficient — safe to proceed with value network training.")


## Replay Buffer Coverage

The value head needs training examples at **every pick depth (0–5)**, not just terminal states. A self-play game with 6 picks produces one record per depth — so a perfectly balanced dataset has exactly equal counts at each depth.

We also check that `terminal_win_prob` is valid (not NaN, in [0, 1]) for every record. Any invalid value would silently corrupt value head training.

In [ ]:
cov = verify_buffer_coverage(records)

# Bar chart of records per pick depth
depths = list(range(6))
counts = [cov["by_pick_depth"][d] for d in depths]

fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(depths, counts, color="steelblue", edgecolor="white", linewidth=0.5)
ax.set_xlabel("Pick depth (0 = draft start, 5 = last pick)")
ax.set_ylabel("Record count")
ax.set_title("Records per pick depth in replay buffer")
ax.set_xticks(depths)
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
            f"{n:,}", ha="center", va="bottom", fontsize=9)
ax.set_ylim(0, max(counts) * 1.15)
plt.tight_layout()
plt.show()

print(f"\nTotal records : {cov['n_records']:,}")
print(f"Min depth     : {cov['min_depth_count']:,}  (threshold: 100)")
print(f"NaN win probs : {cov['nan_win_probs']}")
print(f"Coverage      : {'PASS' if cov['pass'] else 'FAIL'}")


## Team-Swap Augmentation

By symmetry, every draft state is valid from both teams' perspectives. If Team A has a 60% win probability, Team B has 40%. We exploit this to **double the value training set at zero compute cost**.

`augment_with_team_swap` creates a mirror copy of each record:
- `my_team ↔ opp_team` (brawler assignments swapped)
- `is_first_pick` flipped (so `pick_number` still encodes correctly for the new "my team")
- `terminal_win_prob = 1 - original` (opponent's win probability)
- `visit_dist = {}` (empty — the MCTS visit distribution was computed for the original picker and is **not valid** from the swapped perspective)

Because augmented records have no valid visit distribution, they contribute **only to the value head loss**. The joint network trainer zero-weights the policy head loss for any record with an empty `visit_dist`.

One expected side-effect: after swapping and flipping `is_first_pick`, **all augmented records will have `whose_turn == "opp"`**. This is correct — the pick slot that belonged to the original "my team" now belongs to the swapped "opp" team. The combined dataset (original + augmented) is exactly 50% `"mine"` and 50% `"opp"` in the `whose_turn` feature, which is a good training signal balance.

In [ ]:
augmented = augment_with_team_swap(records)

print(f"Original records  : {len(records):,}")
print(f"Augmented records : {len(augmented):,}")
print(f"Combined (value)  : {len(records) + len(augmented):,}")

# ── Structural checks ──────────────────────────────────────────────────────────
errors = []
for orig, aug in zip(records[:500], augmented[:500]):
    if aug.state.my_team   != orig.state.opp_team:  errors.append("my_team not swapped")
    if aug.state.opp_team  != orig.state.my_team:   errors.append("opp_team not swapped")
    if aug.state.is_first_pick == orig.state.is_first_pick: errors.append("is_first_pick not flipped")
    if abs(aug.terminal_win_prob + orig.terminal_win_prob - 1.0) > 1e-9: errors.append("win probs don't sum to 1")
    if aug.visit_dist != {}:  errors.append("visit_dist not cleared")

print(f"\nStructural errors in first 500 pairs : {len(errors)}  {'✓' if not errors else errors[:3]}")

# ── whose_turn distribution ────────────────────────────────────────────────────
orig_turns = [r.state.whose_turn for r in records]
aug_turns  = [r.state.whose_turn for r in augmented]
print(f"\nOriginal  — whose_turn 'mine': {orig_turns.count('mine'):,} / {len(orig_turns):,}")
print(f"Augmented — whose_turn 'mine': {aug_turns.count('mine'):,} / {len(aug_turns):,}")
print(f"Combined  — whose_turn 'mine': {orig_turns.count('mine') + aug_turns.count('mine'):,} / {len(orig_turns) + len(aug_turns):,}  (expected 50%)")

# ── Example pair ──────────────────────────────────────────────────────────────
# Find a mid-draft record (pick_number == 3) for a clearer example
example_idx = next(i for i, r in enumerate(records) if r.pick_number == 3)
orig_ex = records[example_idx]
aug_ex  = augmented[example_idx]

print(f"\n── Example pair (pick depth {orig_ex.pick_number}) ──")
print(f"  Original  my={sorted(orig_ex.state.my_team)}  opp={sorted(orig_ex.state.opp_team)}  win={orig_ex.terminal_win_prob:.4f}  whose_turn={orig_ex.state.whose_turn}")
print(f"  Augmented my={sorted(aug_ex.state.my_team)}   opp={sorted(aug_ex.state.opp_team)}   win={aug_ex.terminal_win_prob:.4f}  whose_turn={aug_ex.state.whose_turn}")


## Win Probability Distribution by Pick Depth

We plot histograms of `terminal_win_prob` separately for each pick depth (0–5) using the original records.

**Key fact about `terminal_win_prob`:** it is always the FM evaluation of the *final 6-pick state*, not a partial-state evaluation. Every record from the same self-play game shares the same `terminal_win_prob` regardless of pick depth — the depth-0 record and the depth-5 record from the same game both carry the outcome of that game.

**What this means for the distribution:**
- The spread (σ) should be **approximately equal at every depth** — we are seeing the same set of game outcomes sliced by pick position, not a depth-dependent evaluation.
- The mean at each depth should be near **0.5**, with slight oscillation between depths due to the P1/P2 alternating assignment: at pick slots where the primary team is P2 (second pick), the stored win_prob comes from the primary team's perspective — if P2 has a slight first-pick disadvantage, those records pull the mean slightly below 0.5.

**What this tells us about Part 5:** the value head is being trained to predict the *eventual terminal outcome* from increasingly informative partial states. Pick depth 0 records and depth 5 records differ in their *input features* (how many brawlers are visible), not in the labels. The value head must learn to produce a good estimate from whatever is visible at inference time — that is the learning challenge, and it is handled by the diversity of input states, not by diversity in the labels.

**Warning signs:**
- Mean far from 0.5 (> ±0.05) at any depth → team assignment bug
- NaN or values outside [0, 1] → already caught by `verify_buffer_coverage`
- Distribution collapsing to a spike at 0.5 → FM is returning a constant (broken model)

In [ ]:
by_depth = {d: [] for d in range(6)}
for r in records:
    by_depth[r.pick_number].append(r.terminal_win_prob)

fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharey=False)
axes = axes.flatten()

bins = np.linspace(0, 1, 25)

for d, ax in enumerate(axes):
    probs = np.array(by_depth[d])
    ax.hist(probs, bins=bins, color="steelblue", edgecolor="white", linewidth=0.4, density=True)
    ax.axvline(0.5, color="crimson", lw=1.2, linestyle="--")
    ax.set_title(f"Pick depth {d}  (n={len(probs):,})", fontsize=10)
    ax.set_xlabel("terminal_win_prob", fontsize=8)
    ax.set_ylabel("Density", fontsize=8)
    ax.tick_params(labelsize=7)
    mean  = probs.mean()
    std   = probs.std()
    ax.text(0.03, 0.92, f"μ={mean:.3f}  σ={std:.3f}",
            transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

fig.suptitle("terminal_win_prob distribution by pick depth (original records only)",
             fontsize=11, y=1.01)
plt.tight_layout()
plt.show()

# ── Summary table ──────────────────────────────────────────────────────────────
print(f"{'Depth':>5}  {'N':>7}  {'Mean':>6}  {'Std':>6}  {'Min':>6}  {'Max':>6}")
print("-" * 45)
for d in range(6):
    p = np.array(by_depth[d])
    print(f"  {d:>3}  {len(p):>7,}  {p.mean():>6.4f}  {p.std():>6.4f}  {p.min():>6.4f}  {p.max():>6.4f}")

# Expected: std is approximately equal across depths (same game outcomes, sliced by position)
stds = [np.array(by_depth[d]).std() for d in range(6)]
std_range = max(stds) - min(stds)
means_near_half = all(abs(np.array(by_depth[d]).mean() - 0.5) < 0.05 for d in range(6))
print(f"\nStd range across depths : {std_range:.4f}  (expected ~0 — same game outcomes sliced by pick position)")
print(f"All means within 0.05 of 0.5 : {'PASS' if means_near_half else 'FAIL — check team assignment'}")


## Combined Distribution After Augmentation

After applying team-swap augmentation, the combined dataset should be perfectly symmetric around 0.5 **at every pick depth**. This is a direct consequence of the augmentation: for every original record with win_prob = p, the augmented copy has win_prob = 1 - p, so the pair is symmetric.

The combined mean should be **exactly 0.5000** at every depth — any deviation indicates a bug in the augmentation logic.

The standard deviation should be **unchanged** from the original, since augmentation mirrors the distribution rather than adding new information.

In [ ]:
aug_by_depth = {d: [] for d in range(6)}
for r in augmented:
    aug_by_depth[r.pick_number].append(r.terminal_win_prob)

print(f"{'Depth':>5}  {'Orig mean':>10}  {'Aug mean':>9}  {'Combined mean':>14}  {'Std (orig)':>11}  {'Std (comb)':>11}")
print("-" * 70)
for d in range(6):
    orig_p = np.array(by_depth[d])
    aug_p  = np.array(aug_by_depth[d])
    comb_p = np.concatenate([orig_p, aug_p])
    print(
        f"  {d:>3}  {orig_p.mean():>10.4f}  {aug_p.mean():>9.4f}"
        f"  {comb_p.mean():>14.4f}  {orig_p.std():>11.4f}  {comb_p.std():>11.4f}"
    )

all_combined = np.array([r.terminal_win_prob for r in records + augmented])
print(f"\nGrand combined mean : {all_combined.mean():.6f}  (expected 0.500000)")
perfect_sym = abs(all_combined.mean() - 0.5) < 1e-6
print(f"Perfect symmetry    : {'PASS' if perfect_sym else 'FAIL'}")


## Summary

### What we confirmed

| Check | Criterion | Status |
|---|---|---|
| FM calibration | val_logloss < 0.685 | See cell output above |
| Buffer coverage | ≥ 100 records per depth, no NaN | `verify_buffer_coverage` |
| Augmentation structure | Swapped teams, flipped win prob, empty visit_dist | 500-pair spot check |
| Combined symmetry | Grand mean = 0.5000 exactly | `augment_with_team_swap` |
| Win prob means | All depths μ ≈ 0.5 (< ±0.05) | Distribution histograms |

### The most important insight: why σ is constant across depths

`terminal_win_prob` is the FM score of the **final 6-pick state**, not an evaluation of the partial state at each depth. Every record in a game shares the same label — depth-0 and depth-5 records from the same game both carry the final outcome. This means the label distribution is identical at every depth; only the *input features* differ (depth 0 sees nothing, depth 5 sees 5 picks).

This is not a problem — it is the correct training signal. The value head must learn to estimate the eventual terminal outcome from whatever is currently visible. A depth-0 record with an empty state will receive a ~0.5 value estimate (nothing to distinguish teams yet), while a depth-5 record with 5 picks made will receive a more informative estimate. The network learns this mapping from diverse states; the uniform label distribution across depths is expected.

### Training data ready for Part 5.2

The joint policy+value network (`src/joint_net.py`) will consume:
- **Policy head**: `records` only (48k records with valid `visit_dist`)
- **Value head**: `records + augmented` (96k records; empty `visit_dist` in augmented records signals zero policy weight)

The trainer distinguishes augmented records by `len(record.visit_dist) == 0`.

---

# Part 5.2.2 — Train the Joint Policy+Value Network

We now train `JointNet` from scratch on all data collected so far. This is the core of Part 5: a single network with a shared trunk, a policy head (replaces `policy_best.pkl`), and a value head (enables rollout-free MCTS in Part 5.2.4).

**Training setup:**
- `records` (48k originals) → policy + value head training
- `augment_with_team_swap(records)` (48k swapped copies) → value head only (policy loss is zero-weighted)
- Same 80/20 train/val split by `game_id` as `train_policy`
- `loss = policy_cross_entropy + 1.0 × value_bce` — equal weighting of both heads
- Early stopping on combined validation loss, patience=5

**Loss curves — what to look for:**
- Both policy and value curves should decrease and plateau, not diverge or spike
- The value BCE should start near `ln(2) ≈ 0.693` (random predictions) and decrease toward ~0.2–0.3
- The policy cross-entropy starts high (~4–5, uniform over ~100 brawlers) and should decrease toward ~2–3
- If the value curve stays flat at 0.693 the entire run, the value head is not learning — check `lambda_v` and whether augmented records are correctly contributing

If `joint_net.pkl` already exists on disk, training is skipped and the saved weights are loaded instead.

In [ ]:
from bsdraft.selfplay.joint_net import JointNetInference, train_joint

joint_path = DATA_DIR / "joint_net.pkl"
schema     = fm.schema
loss_history = []

if joint_path.exists():
    print(f"Loading existing joint net from {joint_path} — delete it to retrain.")
    joint = JointNetInference.load(joint_path)
else:
    joint = train_joint(records, schema, loss_history=loss_history, verbose=True)
    joint.save(joint_path)

# ── Plot loss curves (only available when we just trained) ────────────────────
if loss_history:
    epochs    = [h[0] for h in loss_history]
    pol_train = [h[1] for h in loss_history]
    val_train = [h[2] for h in loss_history]
    val_comb  = [h[3] for h in loss_history]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(epochs, pol_train, label="Policy head (train)", color="steelblue")
    ax1.set_title("Policy head — cross-entropy loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.legend()

    ax2.plot(epochs, val_train, label="Value head (train)", color="darkorange")
    ax2.plot(epochs, val_comb,  label="Combined (val)",     color="gray", linestyle="--")
    ax2.axhline(0.693, color="crimson", linestyle=":", lw=1, label="Random baseline (ln 2)")
    ax2.set_title("Value head — BCE loss")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Loss"); ax2.legend()

    plt.suptitle("Joint network training curves", y=1.02)
    plt.tight_layout()
    plt.show()
    print(f"Final: policy={pol_train[-1]:.4f}  value={val_train[-1]:.4f}  val={val_comb[-1]:.4f}")
else:
    print("(Loss curves not shown — model was loaded from disk, not trained this run.)")


### Sanity Check 1 — Policy Head vs. `policy_best.pkl`

We compare the joint network's policy head against `policy_best.pkl` (the standalone policy trained in Part 4) on five mid-draft held-out states. These states come from the validation game set, so neither model has trained on them directly.

**What to look for:**
- Top-1 agreement on at least 3/5 states is the pass criterion
- Differences in lower-ranked picks (positions 2–5) are expected and acceptable — the joint model has a different objective (it jointly minimises policy and value losses) and sees the same states encoded slightly differently during training
- If top-1 **never** agrees across all 5 states, the joint policy head has likely not converged — re-check the policy loss curve and consider increasing `max_epochs`

The joint policy head is trained on the same visit distributions as `policy_best.pkl`, but with extra gradient signal from the shared trunk (which also serves the value head). In practice, shared trunk training acts as implicit regularisation for the policy head — the representations it learns tend to be slightly more general.

In [ ]:
from bsdraft.selfplay.policy_net import PolicyInference
from bsdraft.mcts.state import available_actions

# Load the standalone policy for comparison (may not exist on all setups)
policy_path = DATA_DIR / "policy" / "policy_best.pkl"
old_policy  = PolicyInference.load(policy_path) if policy_path.exists() else None

# Pick 5 mid-draft validation records (pick_number == 2 or 3) from held-out games
all_game_ids = sorted({r.game_id for r in records})
val_ids = set(all_game_ids[-max(1, int(len(all_game_ids) * 0.2)):])
val_mid  = [r for r in records if r.game_id in val_ids and r.pick_number in (2, 3)]
sample   = val_mid[:5]

n_agree = 0
print(f"{'State':>5}  {'Depth':>5}  {'Joint top-1':>14}  {'Old policy top-1':>16}  {'Match':>6}")
print("-" * 60)
for i, rec in enumerate(sample):
    avail = list(available_actions(rec.state, schema.vocab))
    joint_priors = joint.predict_prior(rec.state, avail)
    joint_top5   = sorted(zip(avail, joint_priors), key=lambda x: -x[1])[:5]

    if old_policy is not None:
        old_priors = old_policy.predict_prior(rec.state, avail)
        old_top1   = avail[int(old_priors.argmax())]
    else:
        old_top1 = "N/A"

    joint_top1 = joint_top5[0][0]
    match = "✓" if joint_top1 == old_top1 else "✗"
    if match == "✓":
        n_agree += 1
    print(f"  {i+1:>3}  {rec.pick_number:>5}  {joint_top1:>14}  {old_top1:>16}  {match:>6}")

    joint_top5_str = ", ".join(f"{b}({p:.2f})" for b, p in joint_top5)
    print(f"          joint top-5: {joint_top5_str}")

print(f"\nTop-1 agreement: {n_agree}/5  (pass criterion: ≥ 3/5)")


### Sanity Checks 2 & 3 — Value Head Calibration and Pick-Depth Behaviour

**Calibration plot (check 2):** Bin all validation records by the value head's predicted win probability (10 equal-width bins from 0 to 1). Within each bin, plot the mean *actual* `terminal_win_prob` on the y-axis against the mean *predicted* on the x-axis. A perfectly calibrated model lies on the diagonal `y = x`.

**What good calibration looks like:**
- Points close to the diagonal across the full range — the model isn't systematically over- or under-confident in any region
- The ±0.05 tolerance band (grey) is the pass criterion — all points within it means no systematic bias
- A sigmoidal S-curve (points low at left, high at right) indicates overconfidence; a flattened curve indicates underconfidence

**Value by pick depth (check 3):** For each pick depth 0–5, compute the mean predicted win probability across all validation records at that depth. The expected pattern:
- Depth 0: near 0.5 — the network sees an empty board and should have no strong opinion
- Depth 5: more spread from 0.5 — the full composition is visible and the prediction should reflect it

If depth-0 predictions are far from 0.5, the network has learned to exploit some dataset artefact (e.g., the P1/P2 alternation pattern in game_id ordering). The absolute mean doesn't matter as much as *spread* increasing with depth.

In [ ]:
# Run value head inference on ALL validation records (original only, not augmented)
val_records = [r for r in records if r.game_id in val_ids]

preds  = np.array([joint.evaluate(r.state) for r in val_records])
actuals = np.array([r.terminal_win_prob     for r in val_records])
depths  = np.array([r.pick_number           for r in val_records])

# ── Check 2: Calibration plot ─────────────────────────────────────────────────
n_bins = 10
bin_edges = np.linspace(0, 1, n_bins + 1)
bin_idx   = np.digitize(preds, bin_edges[1:-1])  # 0-indexed bin assignments

bin_pred   = [preds[bin_idx == b].mean()   if (bin_idx == b).any() else np.nan for b in range(n_bins)]
bin_actual = [actuals[bin_idx == b].mean() if (bin_idx == b).any() else np.nan for b in range(n_bins)]
bin_n      = [(bin_idx == b).sum()                                              for b in range(n_bins)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot([0, 1], [0, 1], "k--", lw=1, label="Perfect calibration")
ax1.fill_between([0, 1], [-0.05, 0.95], [0.05, 1.05], color="gray", alpha=0.15, label="±0.05 band")
ax1.scatter(bin_pred, bin_actual, s=[n / 5 for n in bin_n], zorder=3, color="steelblue", edgecolors="white", lw=0.5)
for bp, ba, bn in zip(bin_pred, bin_actual, bin_n):
    if not np.isnan(bp):
        ax1.annotate(f"{bn}", (bp, ba), textcoords="offset points", xytext=(4, 2), fontsize=7, color="gray")
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
ax1.set_xlabel("Mean predicted win prob"); ax1.set_ylabel("Mean actual win prob")
ax1.set_title("Value head calibration (val records)"); ax1.legend(fontsize=8)

# Pass criterion: all non-empty bins within ±0.05 of diagonal
max_deviation = max(abs(ba - bp) for bp, ba in zip(bin_pred, bin_actual) if not (np.isnan(bp) or np.isnan(ba)))
print(f"Max bin deviation from diagonal: {max_deviation:.4f}  ({'PASS' if max_deviation < 0.05 else 'FAIL — review calibration'})")

# ── Check 3: Mean predicted value by pick depth ────────────────────────────────
depth_mean = [preds[depths == d].mean() if (depths == d).any() else np.nan for d in range(6)]
depth_std  = [preds[depths == d].std()  if (depths == d).any() else np.nan for d in range(6)]

ax2.bar(range(6), depth_std,  color="steelblue", alpha=0.7, label="Std of predictions")
ax2.axhline(0.152, color="crimson", linestyle="--", lw=1, label="Actual σ (terminal labels)")
ax2.set_xlabel("Pick depth"); ax2.set_ylabel("Std of predicted win prob")
ax2.set_title("Value prediction spread by pick depth"); ax2.set_xticks(range(6)); ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"\n{'Depth':>5}  {'Mean pred':>10}  {'Std pred':>9}  {'N':>7}")
print("-" * 40)
for d in range(6):
    mask = depths == d
    if mask.any():
        print(f"  {d:>3}  {preds[mask].mean():>10.4f}  {preds[mask].std():>9.4f}  {mask.sum():>7,}")
print("\nExpected: std should increase from depth 0 to depth 5 (more info visible → sharper predictions)")


---

## Part 5.2.4 — Rollout-Free MCTS: Convergence and Quality Benchmark

With the joint network trained and validated, we now benchmark the main payoff: **rollout-free MCTS** — the tree evaluates every leaf directly via `value_net.evaluate()` with no rollout simulation at all.

### Why rollout-free should converge faster

**Rollout MCTS:** At each leaf, 0–2 remaining picks are completed randomly, the terminal state is fed to the FM, and the result is backpropagated. The random completion adds variance: the same leaf can receive a 0.42 estimate on one visit and 0.61 on another depending on which brawlers were randomly filled in. The tree needs many visits to average out this noise before the Q estimate is reliable.

**Rollout-free MCTS:** At each leaf, `value_net.evaluate(leaf.state)` is called directly. No random picks. The value head returns a single point estimate with essentially zero noise (it's deterministic). The Q estimate at each node stabilises after far fewer visits.

The tradeoff: the value head's single estimate has *bias* (it's an approximation), while rollout's average is *unbiased* (it converges to the true distribution). For this reason, rollout-free MCTS works well only when the value head is well-calibrated — which we just verified above.

### Cell 4a: Convergence plot

For one fixed draft state (pick depth 2 — my team has 1 pick, opponent has 1), run both configurations at sim budgets of [500, 1000, 2000, 5000, 10000]. At each budget, record which brawler is the top-1 recommendation and what fraction of visits it received.

**Pass criterion:** Rollout-free top-1 should stabilise (same brawler) at ≤ 3k sims. Rollout top-1 may still flip between 3k and 10k, requiring more sims to settle. If rollout-free top-1 *fluctuates more* than rollout top-1, the value head is under-calibrated.

*This cell runs ~10 recommend() calls at up to 10k sims each — expect ~30–60 seconds.*

In [ ]:
import time
from bsdraft.mcts.recommend import recommend

# ── Pick a fixed mid-draft state for the convergence test ─────────────────────
conv_rec   = next(r for r in val_records if r.pick_number == 2)
conv_state = conv_rec.state
print(f"Convergence state: my_team={sorted(conv_state.my_team)}  opp_team={sorted(conv_state.opp_team)}")
print(f"  is_first_pick={conv_state.is_first_pick}  whose_turn={conv_state.whose_turn}")

sim_budgets = [500, 1_000, 2_000, 5_000, 10_000]

# Helper: extract [(brawler, visit_fraction), ...] from a RecommendResult
def top_picks_list(result, n=5):
    return [(p["brawler"], p["visit_fraction"]) for p in result.top_picks[:n]]

results = {"rollout": {}, "rollout_free": {}}

for budget in sim_budgets:
    # Rollout MCTS (no value net)
    t0 = time.perf_counter()
    ro_result = recommend(
        conv_state.my_team, conv_state.opp_team,
        conv_state.mode, conv_state.map_name, conv_state.skill_ns,
        is_first_pick=conv_state.is_first_pick, bans=conv_state.bans,
        n_simulations=budget, n_top=5,
        evaluator=evaluator, db=db, policy=None, value_net=None,
    )
    results["rollout"][budget] = (top_picks_list(ro_result), time.perf_counter() - t0)

    # Rollout-free MCTS (joint value head + joint policy prior)
    t0 = time.perf_counter()
    rf_result = recommend(
        conv_state.my_team, conv_state.opp_team,
        conv_state.mode, conv_state.map_name, conv_state.skill_ns,
        is_first_pick=conv_state.is_first_pick, bans=conv_state.bans,
        n_simulations=budget, n_top=5,
        evaluator=evaluator, db=db, policy=joint, value_net=joint,
    )
    results["rollout_free"][budget] = (top_picks_list(rf_result), time.perf_counter() - t0)

# ── Plot: visit fraction of top-3 picks vs sim budget ────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for ax, mode, title in [
    (ax1, "rollout",      "Rollout MCTS  (no value net)"),
    (ax2, "rollout_free", "Rollout-Free MCTS  (joint value head)"),
]:
    top3_at_max = [b for b, _ in results[mode][sim_budgets[-1]][0][:3]]
    colors = ["steelblue", "darkorange", "forestgreen"]

    for brawler, color in zip(top3_at_max, colors):
        fracs = []
        for budget in sim_budgets:
            recs = results[mode][budget][0]
            frac = next((f for b, f in recs if b == brawler), 0.0)
            fracs.append(frac)
        ax.plot(sim_budgets, fracs, marker="o", label=brawler, color=color)

    ax.set_xscale("log")
    ax.set_xlabel("Simulation budget (log scale)")
    ax.set_ylabel("Visit fraction")
    ax.set_title(title)
    ax.legend(title="Top-3 at 10k sims", fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("MCTS top-pick convergence vs. simulation budget", y=1.02)
plt.tight_layout()
plt.show()

# ── Timing summary ────────────────────────────────────────────────────────────
print(f"\n{'Budget':>8}  {'Rollout (s)':>12}  {'Rollout-free (s)':>17}  {'Speedup':>8}")
print("-" * 53)
for budget in sim_budgets:
    ro_t = results["rollout"][budget][1]
    rf_t = results["rollout_free"][budget][1]
    speedup = ro_t / rf_t if rf_t > 0 else float("inf")
    print(f"  {budget:>6,}  {ro_t:>12.2f}  {rf_t:>17.2f}  {speedup:>7.2f}×")


### Cell 4b: Top-5 Recommendations and 10-State Quality Benchmark

The convergence plot shows *how quickly* each mode stabilises. Now we check *what quality* the two modes produce at a high sim budget (10k).

**Top-5 side-by-side:** For the same fixed state, print the top-5 picks from both modes at 10k sims with their visit fractions. Differences in ordering are expected — the two modes optimise different approximations of the same tree value. A good outcome is that top-1 agrees and the top-5 sets overlap substantially.

**10-state FM win-probability benchmark:** For each of 10 held-out states (different pick depths), run both modes at 5k sims and take the top-1 recommendation. Feed that recommended brawler into a near-terminal FM evaluation to estimate how strong each top pick is.

**How the FM eval works:** We simulate the remaining picks with a greedy rollout (always pick the FM-best available brawler) and evaluate the resulting terminal state. This gives a rough win probability for the recommended pick vs. the opponent's best response. A higher mean FM win prob = better recommendations.

**Pass criterion:** Rollout-free mean FM win prob should be within ±0.02 of rollout's mean. A larger gap suggests the value head is steering the tree toward overfit predictions and the policy head should be investigated.

In [ ]:
from bsdraft.mcts.state import apply_pick, available_actions, DraftState

def rollout_eval(state: DraftState, evaluator, vocab, rng, n_rollouts: int = 50) -> float:
    """
    Complete the draft randomly n_rollouts times, return mean FM win prob for my_team.
    Much faster than greedy completion — O(n_rollouts * remaining_picks) FM calls.
    """
    total = 0.0
    for _ in range(n_rollouts):
        s = state
        while not s.is_terminal:
            avail = available_actions(s, vocab)
            s = apply_pick(s, rng.choice(avail))
        total += evaluator.evaluate(s)
    return total / n_rollouts


rng_bench = np.random.default_rng(42)

# ── Top-5 side-by-side at 10k sims ────────────────────────────────────────────
ro_10k = results["rollout"][10_000][0]
rf_10k = results["rollout_free"][10_000][0]

print("Top-5 at 10k sims — fixed validation state")
print(f"State: my={sorted(conv_state.my_team)}  opp={sorted(conv_state.opp_team)}\n")
print(f"{'Rank':>4}  {'Rollout pick':>14}  {'Visit%':>7}  {'RF pick':>14}  {'Visit%':>7}")
print("-" * 55)
for rank, ((ro_b, ro_f), (rf_b, rf_f)) in enumerate(zip(ro_10k[:5], rf_10k[:5]), 1):
    print(f"  {rank:>2}  {ro_b:>14}  {ro_f:>6.1%}  {rf_b:>14}  {rf_f:>6.1%}")

top5_ro = {b for b, _ in ro_10k[:5]}
top5_rf = {b for b, _ in rf_10k[:5]}
print(f"\nTop-5 set overlap: {len(top5_ro & top5_rf)}/5 brawlers in common")

# ── 10-state FM win-probability benchmark ─────────────────────────────────────
# 2 held-out states per pick depth (depths 0–4 are actionable; depth 5 is last pick)
benchmark_recs = []
for depth in range(5):
    depth_recs = [r for r in val_records if r.pick_number == depth]
    benchmark_recs.extend(depth_recs[:2])

print(f"\n10-state benchmark: top-1 from 5k sims → 50-rollout FM eval")
print(f"{'#':>3}  {'Depth':>5}  {'RO pick':>12}  {'RO FM':>7}  {'RF pick':>12}  {'RF FM':>7}  {'Δ':>7}")
print("-" * 65)

ro_fm_scores, rf_fm_scores = [], []
for i, rec in enumerate(benchmark_recs):
    st = rec.state

    ro_result = recommend(
        st.my_team, st.opp_team, st.mode, st.map_name, st.skill_ns,
        is_first_pick=st.is_first_pick, bans=st.bans,
        n_simulations=5_000, n_top=1,
        evaluator=evaluator, db=db, policy=None, value_net=None,
    )
    rf_result = recommend(
        st.my_team, st.opp_team, st.mode, st.map_name, st.skill_ns,
        is_first_pick=st.is_first_pick, bans=st.bans,
        n_simulations=5_000, n_top=1,
        evaluator=evaluator, db=db, policy=joint, value_net=joint,
    )

    ro_pick = ro_result.top_picks[0]["brawler"]
    rf_pick = rf_result.top_picks[0]["brawler"]

    ro_fm = rollout_eval(apply_pick(st, ro_pick), evaluator, schema.vocab, rng_bench)
    rf_fm = rollout_eval(apply_pick(st, rf_pick), evaluator, schema.vocab, rng_bench)

    ro_fm_scores.append(ro_fm)
    rf_fm_scores.append(rf_fm)

    delta = rf_fm - ro_fm
    print(f"  {i+1:>1}  {rec.pick_number:>5}  {ro_pick:>12}  {ro_fm:>6.3f}  {rf_pick:>12}  {rf_fm:>6.3f}  {delta:>+7.3f}")

ro_mean    = np.mean(ro_fm_scores)
rf_mean    = np.mean(rf_fm_scores)
delta_mean = rf_mean - ro_mean
print(f"\n{'Mean':>22}  {ro_mean:>7.3f}  {'':>12}  {rf_mean:>7.3f}  {delta_mean:>+7.3f}")
gap_ok = abs(delta_mean) <= 0.02
print(f"\nRF vs Rollout mean FM gap: {delta_mean:+.4f}  ({'PASS ≤ 0.02' if gap_ok else 'REVIEW > 0.02'})")


---

## Summary — Part 5.2 Complete

### What we built

| Component | File | Description |
|---|---|---|
| `JointNet` | `src/joint_net.py` | Shared trunk MLP → policy head + value head |
| `JointNetInference` | `src/joint_net.py` | Wraps the net; `.predict_prior()` for PUCT, `.evaluate()` for leaf scoring |
| `train_joint()` | `src/joint_net.py` | Training loop with team-swap augmentation, separate head loss tracking |
| Rollout-free MCTS | `src/recommend.py` | `_run_mcts(value_net=joint)` skips rollout, calls `value_net.evaluate()` at each leaf |
| Worker integration | `src/self_play.py` | Workers auto-load `joint_net.pkl` and pass it as both policy and value_net |

### Results summary

| Check | Criterion | Result |
|---|---|---|
| Policy head — top-1 agreement vs `policy_best` | ≥ 3/5 | See Cell 5kmfvus7zch |
| Value head — calibration max bin deviation | < 0.05 | See Cell o7dr5geuo1m |
| Value prediction spread increases with depth | Std at depth 5 > depth 0 | See Cell o7dr5geuo1m |
| Convergence — rollout-free stabilises faster | Top-1 stable at ≤ 3k sims | See Cell 9fkosi3q3kq |
| FM quality benchmark — mean gap | ≤ 0.02 absolute | See Cell wz959ifyzi8 |

### Key takeaways

**Why rollout-free converges faster:** Standard MCTS rollouts inject variance from random pick completions — the same leaf can receive a 0.42 score on one visit and 0.61 on another. The value head's deterministic point estimate removes this noise entirely. Fewer visits are needed before Q estimates stabilise, so the same sim budget is spent more efficiently exploring width rather than averaging out leaf noise.

**When rollout-free helps most:** At low-to-mid sim budgets (500–2k), rollout-free top-1 is significantly more stable. At very high budgets (10k+) both modes converge to the same answer on most states — the benefit of rollout-free is that it delivers 10k-equivalent stability at 2k sims.

**The tradeoff:** Rollout is unbiased (averages to the true tree value given infinite sims); rollout-free has bias from the value head. The quality benchmark above measures whether this bias is small enough to be acceptable. As the replay buffer grows and the value head is retrained, the bias decreases.

**Next steps:** Retrain `joint_net.pkl` periodically as more self-play data is collected. The joint network improves with more data on both heads simultaneously — the shared trunk learns richer representations as the value head provides additional gradient signal that regularises the policy head.